# 🧠 Da7ee7-El-Dof3a — Kaggle AI Service

This notebook is the full **AI Service** for Da7ee7-El-Dof3a (دحيح الدفعة).

It runs on a Kaggle GPU instance and exposes a FastAPI server through an
**ngrok** tunnel, which the local backend calls for all heavy AI work:

1. Install dependencies & detect GPU
2. Compare 5 candidate LLMs and auto-select the best one
3. Apply BitsAndBytes quantization (4-bit NF4, falling back to 8-bit)
4. Load documents (PDF / PPTX / DOCX), transcribe audio & video with Whisper
5. Chunk text, build embeddings, index with FAISS
6. Build a LangChain RetrievalQA pipeline
7. Generate Smart Summary, Important Topics, Solved Exams, Revision Notes
8. Export results as PDF
9. Expose everything via FastAPI + ngrok

> **Before running:** In Kaggle → *Add-ons → Secrets*, add a secret named
> `NGROK_AUTH_TOKEN` with your ngrok auth token. Also enable **GPU T4 x2**
> (or better) under *Settings → Accelerator*.

In [ ]:
!pip show numpy
!pip show torch
!pip show transformers

Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the names of any
       contributors may be used to endorse or promote products derived
       from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYR

In [ ]:
!pip install -q \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-core==0.2.38 \
langchain-text-splitters==0.2.4 \
langchain-huggingface==0.0.3

In [ ]:
!pip install -q python-pptx python-docx pypdf

In [ ]:
!pip install -q openai-whisper

In [ ]:
!pip install -q reportlab

In [ ]:
!pip install -q pyngrok

In [ ]:
!pip install -U bitsandbytes accelerate

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.6 MB/s eta 0:00:00:00:0100:01


In [ ]:
# =============================================================================
# 2. IMPORT LIBRARIES
# =============================================================================

import os
import gc
import re
import json
import time
import shutil
import logging
import threading
import tempfile
from pathlib import Path
from typing import Optional, Any
from xml.sax.saxutils import escape as xml_escape

import torch
import numpy as np

# ==========================
# Hugging Face
# ==========================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline as hf_pipeline,
)

# ==========================
# LangChain
# ==========================
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_community.vectorstores import FAISS

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFacePipeline,
)
# NOTE: `langchain.chains.RetrievalQA` was imported in the original notebook but
# never actually used (summaries were generated manually via run_llm()). Removed
# as dead code — the summarization pipeline below is a hand-rolled Map-Reduce
# chain instead, which gives us full control over chunk batching and logging.

# ==========================
# Document Processing
# ==========================
from pypdf import PdfReader
from pptx import Presentation
from docx import Document as DocxDocument

# ==========================
# Audio / Video
# ==========================
import whisper
try:
    from moviepy import VideoFileClip          # moviepy >= 2.0
except ImportError:
    from moviepy.editor import VideoFileClip   # moviepy < 2.0 fallback

# ==========================
# PDF Export
# ==========================
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    ListFlowable,
    ListItem,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# ==========================
# FastAPI
# ==========================
from fastapi import FastAPI, UploadFile, File, Form, Request
from fastapi.responses import FileResponse, JSONResponse

import uvicorn
import nest_asyncio

# ==========================
# ngrok
# ==========================
from pyngrok import ngrok, conf as ngrok_conf

nest_asyncio.apply()

# =============================================================================
# LOGGING — every stage below (ingest, indexing, retrieval, generation, PDF
# export, downloads) logs through this logger so failures are traceable
# instead of silently returning bad output.
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("da7ee7_ai_service")
logger.info("All libraries imported successfully.")


error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evalu

In [ ]:
# 3. DETECT GPU
# -----------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    logger.info(f"GPU detected: {gpu_name} ({total_mem_gb:.1f} GB)")
else:
    logger.warning("No GPU detected. Model comparison / inference will be very slow.")

WORKDIR = Path("/kaggle/working/da7ee7")
UPLOADS_DIR = WORKDIR / "uploads"
INDEX_DIR = WORKDIR / "faiss_indexes"
OUTPUT_DIR = WORKDIR / "outputs"
for d in (UPLOADS_DIR, INDEX_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"Working directories ready under {WORKDIR}")


2026-07-30 13:25:23 | INFO     | da7ee7_ai_service | GPU detected: Tesla T4 (14.6 GB)
2026-07-30 13:25:23 | INFO     | da7ee7_ai_service | Working directories ready under /kaggle/working/da7ee7


## 4. Load the LLM — one fixed, free, open-weight model (no per-session benchmarking)

The previous version loaded **5 different candidate models** back-to-back
(Qwen2.5-7B, Gemma-3-4b, Llama-3.2-3B, Phi-4-mini, Mistral-7B) just to score
them on 3 toy prompts and pick a "winner". In practice this was:

- **Slow** — spent most of the Kaggle GPU-hour budget loading/discarding
  models before any real summarization/exam work started.
- **Unreliable** — the cheap heuristic scorer (word-uniqueness + length) is
  not a real quality metric. In production it sometimes selected a smaller
  model (e.g. Phi-4-mini) that then hallucinated invented references,
  leaked raw chat/special tokens (`<|end_of_document|>`) into the output,
  and repeated entire paragraphs verbatim — none of which the benchmark
  step could have caught with 3 short prompts.
- **Fragile** — Llama-3.2 and Gemma-3 are gated on Hugging Face and require
  an accepted license + `HF_TOKEN`; a cold Kaggle session without that
  token silently lost 2 of the 5 candidates every run.

Instead, we hardcode **one** well-regarded, fully open instruction model —
no license gate, no benchmarking loop, one model load:

**`Qwen/Qwen2.5-7B-Instruct`**

- Apache-2.0 licensed, no HF login / gated-repo approval required.
- Strong instruction-following and summarization quality for a 7B model.
- Good multilingual support (Arabic + English), which matters for
  Da7ee7-El-Dof3a's lecture content.
- 32k context window — helpful for the Map-Reduce summarization step.
- Fits comfortably in 4-bit NF4 on a single T4 (~14.6 GB).

In [ ]:
# 4a. Quantization config — prefer 4-bit NF4, fall back to 8-bit if unsupported.
# -----------------------------------------------------------------------------
def build_quant_config() -> tuple[BitsAndBytesConfig, str]:
    try:
        cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        return cfg, "4-bit NF4"
    except Exception as exc:
        logger.warning(f"4-bit NF4 unsupported ({exc}), falling back to 8-bit.")
        cfg = BitsAndBytesConfig(load_in_8bit=True)
        return cfg, "8-bit"


QUANT_CONFIG, QUANT_MODE = build_quant_config()
logger.info(f"Using quantization mode: {QUANT_MODE}")


2026-07-30 13:25:23 | INFO     | da7ee7_ai_service | Using quantization mode: 4-bit NF4


In [ ]:
# 4b. Load the single fixed model + tokenizer.
# -----------------------------------------------------------------------------
def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Several instruction models (Qwen, Llama, Mistral) ship WITHOUT a pad
    # token. Leaving this unset causes noisy "Setting pad_token_id to
    # eos_token_id" warnings on every generation call, and can cause
    # unpredictable batching/attention-mask behavior. Set it once, here.
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=QUANT_CONFIG,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


# The single model used for this service — see the markdown cell above for
# why this replaces the old 5-model benchmarking step.
SELECTED_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

logger.info(f"Loading fixed model: {SELECTED_MODEL_NAME} (quantization: {QUANT_MODE})")
llm_model, llm_tokenizer = load_model_and_tokenizer(SELECTED_MODEL_NAME)
logger.info(f"Model loaded: {SELECTED_MODEL_NAME}")


2026-07-30 13:25:23 | INFO     | da7ee7_ai_service | Loading fixed model: Qwen/Qwen2.5-7B-Instruct (quantization: 4-bit NF4)
2026-07-30 13:25:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-30 13:25:23 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-30 13:25:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-07-30 13:25:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-30 13:25:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-In

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

2026-07-30 13:26:01 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-30 13:26:01 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"
2026-07-30 13:26:01 | INFO     | da7ee7_ai_service | Model loaded: Qwen/Qwen2.5-7B-Instruct


In [ ]:
# 4c. Generation config — this is the main fix for hallucinated / repeated /
# looping output:
#   - do_sample=False, num_beams=1  -> greedy decoding: fully deterministic,
#     removes the random-sampling noise that caused made-up facts.
#   - repetition_penalty=1.25       -> penalizes tokens already generated,
#     directly targets "repeated words" / "repeated sentences".
#   - no_repeat_ngram_size=4        -> hard-blocks any 4-gram from repeating
#     verbatim, which is what stops the endless-loop failure mode outright.
#   - max_new_tokens=300 (within the 250-350 sweet spot) + min_new_tokens=16
#     -> bounded output length, avoids run-on generation.
#   - eos_token_id / pad_token_id set explicitly -> removes the "pad token
#     not set" warning and lets the model stop cleanly at its own EOS.
#   - return_full_text=False        -> the pipeline returns ONLY the newly
#     generated text, so the prompt/instructions never leak into the
#     "summary" (a major source of the earlier hallucination/garbage-output
#     symptom).
#
# NOTE: temperature / top_p are intentionally NOT passed here. They only
# apply when do_sample=True; passing them anyway just produces a
# UserWarning for no benefit, since greedy decoding ignores them.
# -----------------------------------------------------------------------------
GENERATION_KWARGS = dict(
    max_new_tokens=300,
    min_new_tokens=16,
    do_sample=False,
    num_beams=1,
    repetition_penalty=1.25,
    no_repeat_ngram_size=4,
)

text_gen_pipeline = hf_pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    eos_token_id=llm_tokenizer.eos_token_id,
    pad_token_id=llm_tokenizer.pad_token_id,
    return_full_text=False,
    **GENERATION_KWARGS,
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
logger.info(f"LLM ready: {SELECTED_MODEL_NAME} | quantization: {QUANT_MODE} | generation: {GENERATION_KWARGS}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'num_beams', 'eos_token_id', 'no_repeat_ngram_size', 'min_new_tokens', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
2026-07-30 13:26:01 | INFO     | da7ee7_ai_service | LLM ready: Qwen/Qwen2.5-7B-Instruct | quantization: 4-bit NF4 | generation: {'max_new_tokens': 300, 'min_new_tokens': 16, 'do_sample': False, 'num_beams': 1, 'repetition_penalty': 1.25, 'no_repeat_ngram_size': 4}


## 5. Embedding model + Whisper

In [ ]:
# 5a. Embedding model for FAISS (multilingual — handles Arabic/English lecture content).
# -----------------------------------------------------------------------------
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={"device": DEVICE},
)

# 5b. Whisper model for audio/video transcription.
whisper_model = whisper.load_model("medium", device=DEVICE)
logger.info("Embedding model and Whisper model loaded.")


2026-07-30 13:26:02 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-07-30 13:26:02 | INFO     | datasets | JAX version 0.7.2 available.
2026-07-30 13:26:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-30 13:26:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json "HTTP/1.1 200 OK"
2026-07-30 13:26:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-30 13:26:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-30 13:26:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-30 13:26:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-30 13:26:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-30 13:26:06 | INFO     | ht

## 6. Document loaders — PDF, PPTX, DOCX, audio, video

In [ ]:
# 6a. Loader functions for every supported file type.
# -----------------------------------------------------------------------------
def load_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def load_pptx(path: Path) -> str:
    prs = Presentation(str(path))
    chunks = []
    for slide in prs.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text:
                chunks.append(shape.text)
    return "\n".join(chunks)


def load_docx(path: Path) -> str:
    doc = DocxDocument(str(path))
    return "\n".join(p.text for p in doc.paragraphs)


def extract_audio_from_video(path: Path) -> Path:
    audio_path = path.with_suffix(".wav")
    clip = VideoFileClip(str(path))
    clip.audio.write_audiofile(str(audio_path), logger=None)
    clip.close()
    return audio_path


def transcribe_audio(path: Path) -> str:
    result = whisper_model.transcribe(str(path))
    return result.get("text", "")


def clean_text(raw_text: str) -> str:
    text = re.sub(r"\s+", " ", raw_text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    return text.strip()


def load_any_file(path: Path) -> str:
    suffix = path.suffix.lower()
    logger.info(f"Loading '{path.name}' (type={suffix})")
    try:
        if suffix == ".pdf":
            raw = load_pdf(path)
        elif suffix in (".ppt", ".pptx"):
            raw = load_pptx(path)
        elif suffix in (".doc", ".docx"):
            raw = load_docx(path)
        elif suffix in (".mp3", ".wav", ".m4a"):
            raw = transcribe_audio(path)
        elif suffix in (".mp4", ".mov", ".mkv"):
            audio_path = extract_audio_from_video(path)
            raw = transcribe_audio(audio_path)
        else:
            logger.warning(f"Unsupported file type '{suffix}' for '{path.name}' — skipping.")
            raw = ""
    except Exception as exc:
        logger.exception(f"Failed to load/transcribe '{path.name}': {exc}")
        raw = ""

    cleaned = clean_text(raw)
    logger.info(f"Loaded '{path.name}': {len(cleaned)} characters extracted.")
    return cleaned


## 7. Chunking, embeddings, FAISS, and the RetrievalQA chain (per session)

In [ ]:
# 7a. In-memory session registry: session_id -> {retriever, vectorstore, documents, file_texts, filenames}
# -----------------------------------------------------------------------------
SESSIONS: dict[str, dict[str, Any]] = {}

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def build_session_index(session_id: str, file_paths: list[Path]) -> dict[str, Any]:
    """
    Load every uploaded file, chunk it, and build a FAISS index for the
    session. Also keeps:
      - `documents`: the FULL list of chunks (used by Map-Reduce summarization
        instead of a 5-result similarity search, so the whole course is
        actually summarized).
      - `file_texts`: filename -> full raw text (used by solve_exam_file() to
        reliably recover an exam file's full text — the original code tried
        to rebuild this from mismatched raw_texts/filenames lists and by
        reaching into FAISS's private `docstore._dict`, which is why
        solved_exam.pdf sometimes silently failed to generate).
    """
    documents: list[Document] = []
    filenames: list[str] = []
    file_texts: dict[str, str] = {}

    for path in file_paths:
        text = load_any_file(path)
        if not text:
            logger.warning(f"No extractable text in '{path.name}' — excluded from index.")
            continue

        filenames.append(path.name)
        file_texts[path.name] = text

        chunks = text_splitter.split_text(text)
        for chunk in chunks:
            documents.append(Document(page_content=chunk, metadata={"source": path.name}))

    if not documents:
        logger.error(f"Ingest failed for session={session_id}: no extractable text in any uploaded file.")
        raise ValueError("No extractable text found in the uploaded files.")

    logger.info(f"Building FAISS index: {len(documents)} chunks from {len(filenames)} file(s) (session={session_id})")
    vectorstore = FAISS.from_documents(documents, embedding_model)
    vectorstore.save_local(str(INDEX_DIR / session_id))
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    SESSIONS[session_id] = {
        "retriever": retriever,
        "vectorstore": vectorstore,
        "filenames": filenames,
        "documents": documents,      # full chunk list -> Map-Reduce summarization
        "file_texts": file_texts,    # filename -> full text -> solve_exam_file()
    }
    logger.info(f"Session '{session_id}' indexed successfully ({len(documents)} chunks).")
    return SESSIONS[session_id]


def get_session(session_id: str) -> dict[str, Any]:
    if session_id not in SESSIONS:
        logger.error(f"Unknown session_id '{session_id}' requested.")
        raise KeyError(f"Unknown session_id '{session_id}'. Upload files first.")
    return SESSIONS[session_id]


## 8. Prompt templates + generation chains

In [ ]:
# 8a. Prompt templates — rewritten for concise, bullet-point, hallucination-free,
# context-only output. A dedicated CHUNK_SUMMARY_PROMPT drives the Map step of
# Map-Reduce summarization; SMART_SUMMARY_PROMPT now drives the Reduce step.
# -----------------------------------------------------------------------------
CHUNK_SUMMARY_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a, a precise academic teaching assistant.\n"
        "Summarize ONLY the key facts, definitions, formulas, and concepts in "
        "the text below. Use short bullet points. Do NOT repeat any point. "
        "Do NOT add information that is not in the text. Do NOT add an intro "
        "or closing sentence — bullets only.\n\n"
        "Text:\n---------\n{context}\n---------\n\nBullet-point summary:"
    ),
)

SMART_SUMMARY_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a, a precise academic teaching assistant.\n"
        "Below are bullet-point summaries covering every part of a full lecture "
        "course, in no particular order. Merge them into ONE final Smart "
        "Summary that:\n"
        "- Covers every distinct concept exactly once (remove cross-part duplicates)\n"
        "- Is organized in a logical topic order, using bullet points and short headings\n"
        "- Highlights key terms, definitions, and formulas\n"
        "- Uses ONLY the information given below — never invent facts\n"
        "- Has no filler text and no closing remarks\n\n"
        "Partial summaries:\n---------\n{context}\n---------\n\nFinal Smart Summary:"
    ),
)

IMPORTANT_TOPICS_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a. Using ONLY the material below, identify the "
        "most important exam topics, ranked by how heavily they are emphasized. "
        "Return between 5 and 12 short topic names.\n"
        "Return STRICT JSON ONLY — a single JSON array of strings, nothing else, "
        "no markdown, no explanation. Example: [\"Topic A\", \"Topic B\"]\n\n"
        "Material:\n---------\n{context}\n---------\n\nJSON array:"
    ),
)

# -----------------------------------------------------------------------------
# SOLVE_EXAM_PROMPT — rewritten to enforce short, exam-revision-style answers.
# This directly targets the "answers are too long" bug: the previous version
# invited "brief, numbered reasoning steps" with no hard length rule, which
# combined with max_new_tokens=300 reliably produced multi-paragraph answers.
# The prompt now hard-caps every answer at 2-5 lines, forbids intros/closings,
# prefers bullets, and gives the model a one-shot example (per spec) so it has
# a concrete target to imitate. The "[General knowledge]" escape hatch from
# the previous version is REPLACED with the exact required fallback sentence,
# so answers stay grounded in the uploaded material only.
# -----------------------------------------------------------------------------
SOLVE_EXAM_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are Da7ee7-El-Dof3a, answering a university exam question for a student.\n"
        "Follow these rules strictly:\n"
        "- Answer ONLY the question below.\n"
        "- Use ONLY the provided context — never invent facts, never hallucinate.\n"
        "- Keep the ENTIRE answer between 2 and 5 lines. Never exceed 5 lines.\n"
        "- Use simple, direct language suitable for exam revision.\n"
        "- Include only the key points needed for full marks. Do NOT repeat information.\n"
        "- Do NOT write an introduction or a conclusion — go straight to the answer.\n"
        "- Prefer short bullet points (each starting with '-') whenever appropriate.\n"
        "- Stop generating as soon as the answer is complete.\n"
        "- If the context does not contain the answer, respond with EXACTLY this "
        "sentence and nothing else: \"The answer was not found in the uploaded material.\"\n\n"
        "Example:\n"
        "Question: Explain Deadlock.\n"
        "Answer:\n"
        "- Deadlock is a situation where two or more processes wait indefinitely for resources held by each other.\n"
        "- It occurs when none of the processes can continue execution.\n"
        "- The four necessary conditions are Mutual Exclusion, Hold and Wait, No Preemption, and Circular Wait.\n\n"
        "Context:\n---------\n{context}\n---------\n\n"
        "Question:\n{question}\n\nAnswer:"
    ),
)

REVISION_NOTES_PROMPT = PromptTemplate(
    input_variables=["summary", "topics"],
    template=(
        "You are Da7ee7-El-Dof3a. Using ONLY the Smart Summary and Important "
        "Topics below, create a FINAL revision sheet, under 400 words, with "
        "exactly these three sections and nothing else:\n"
        "1. Must-Know (5-10 short bullets)\n"
        "2. Formulas / Definitions (bullets — omit this section if none apply)\n"
        "3. Common Mistakes (2-4 short bullets)\n"
        "No introduction, no closing remarks, no repeated bullets.\n\n"
        "Smart Summary:\n---------\n{summary}\n---------\n\n"
        "Important Topics:\n---------\n{topics}\n---------\n\n"
        "Final Revision Notes:"
    ),
)


# -----------------------------------------------------------------------------
# 8b. LLM call helpers.
# -----------------------------------------------------------------------------
def build_chat_prompt(user_content: str) -> str:
    """
    Wrap a raw instruction string using the model's chat template.

    Every candidate model (Qwen2.5-Instruct, Llama-3.2-Instruct, Gemma-3-it,
    Phi-4-mini-instruct, Mistral-Instruct) is instruction-tuned and expects
    its own special chat markup (<|im_start|>, [INST], etc.). The original
    code fed these models a bare completion-style string, which is the
    single biggest contributor to rambling / hallucinated / looping output —
    instruct models were never trained to free-associate from raw text the
    way base models are. Routing every prompt through apply_chat_template()
    fixes this at the source.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are Da7ee7-El-Dof3a, a precise, factual teaching assistant. "
                "Never repeat sentences or words. Never invent information that "
                "is not in the provided context."
            ),
        },
        {"role": "user", "content": user_content},
    ]
    try:
        return llm_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback for any tokenizer without a chat template configured.
        return user_content


# -----------------------------------------------------------------------------
# EXAM_GENERATION_KWARGS — a separate, SHORTER generation profile for exam
# answers, layered on top of the global GENERATION_KWARGS (cell 16).
#
# Root cause of "answers are too long": every call site previously shared the
# same max_new_tokens=300 budget meant for full summaries. That is far more
# than 2-5 lines of text need, so the model reliably used the whole budget.
#   - max_new_tokens=160   -> hard ceiling well under what 5 lines requires,
#                             enforced independently of prompt-following.
#   - min_new_tokens=8     -> allow very short answers (e.g. the "not found"
#                             fallback sentence) without being padded out.
#   - repetition_penalty=1.3 / no_repeat_ngram_size=3 -> slightly stronger
#     than the summary profile, since short exam answers are exactly where a
#     repetition loop burns through the whole token budget on one repeated
#     bullet.
# do_sample/num_beams are inherited from GENERATION_KWARGS (greedy decoding).
# -----------------------------------------------------------------------------
EXAM_GENERATION_KWARGS = dict(
    GENERATION_KWARGS,
    max_new_tokens=160,
    min_new_tokens=8,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
)


def run_llm(prompt_text: str, **generation_overrides) -> str:
    """
    Run one generation call. Because the pipeline is configured with
    return_full_text=False (see cell 16), `generated_text` already excludes
    the prompt — no manual substring-stripping needed (the old
    `output[len(prompt_text):]` logic broke silently whenever the chat
    template changed the text length, leaking the raw prompt into results).

    `**generation_overrides` (e.g. EXAM_GENERATION_KWARGS) are passed
    straight through to the HF pipeline call, where they override the
    defaults baked in at pipeline-construction time — this is what lets
    exam answers use a much shorter max_new_tokens than summaries/revision
    notes without needing a second pipeline object.
    """
    chat_prompt = build_chat_prompt(prompt_text)
    result = text_gen_pipeline(chat_prompt, **generation_overrides)[0]["generated_text"]
    return result.strip()


def join_context(docs: list[Document], max_chars: int = 6000) -> str:
    joined = "\n\n".join(d.page_content for d in docs)
    return joined[:max_chars]


def batch_documents(documents: list[Document], max_chars: int = 3000) -> list[list[Document]]:
    """Group chunks into batches under `max_chars` so each Map-step LLM call
    gets a bounded, predictable context size."""
    batches: list[list[Document]] = []
    current: list[Document] = []
    current_len = 0
    for doc in documents:
        doc_len = len(doc.page_content)
        if current and current_len + doc_len > max_chars:
            batches.append(current)
            current, current_len = [], 0
        current.append(doc)
        current_len += doc_len
    if current:
        batches.append(current)
    return batches


def parse_json_topics(raw: str) -> list[str]:
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        logger.warning("Important-topics output was not valid JSON — falling back to line-split parsing.")
        return [line.strip("- ").strip() for line in raw.splitlines() if line.strip()][:10]
    try:
        parsed = json.loads(match.group(0))
        return [str(t).strip() for t in parsed if str(t).strip()]
    except json.JSONDecodeError:
        logger.warning("Failed to json.loads() the matched topics array — falling back to line-split parsing.")
        return [line.strip("- ").strip() for line in raw.splitlines() if line.strip()][:10]


# -----------------------------------------------------------------------------
# Question splitting.
#
# FIX (root cause of "treats the entire exam as one single question"): the
# previous pattern only matched a numbering marker when it was *already*
# preceded by a literal newline in the extracted text, and only recognized
# Latin numbering styles (Q1, 1., 1), 1:, 1-). In practice:
#   - A marker at the very start of the extracted text (position 0) has no
#     leading "\n", so it never matched — harmless on its own, but any file
#     whose FIRST question also happened to be its ONLY cleanly-numbered
#     question would fall through to the "<=1 questions" case below.
#   - Exams using Arabic numbering/markers ("١.", "السؤال الأول", "س1:") never
#     matched at all, so the whole file came back as a single chunk.
#   - PDF/PPTX text extraction sometimes drops blank lines between questions
#     entirely, so even correctly Latin-numbered exams could fail to match.
# This version (a) prepends a sentinel newline so a marker at position 0 is
# still detected, (b) recognizes Arabic-Indic digits and common Arabic
# question markers in addition to the original Latin styles, and (c) falls
# back to blank-line / question-mark splitting if the numbered pattern still
# only finds one "question", instead of silently treating the whole exam as
# a single question.
# -----------------------------------------------------------------------------
QUESTION_SPLIT_PATTERN = re.compile(
    r"\n(?=\s*(?:"
    r"Q(?:uestion)?\s*\.?\s*\d+\s*[\.\):\-]?"                       # Q1, Question 1, Q.1
    r"|\d+\s*[\.\):\-]"                                             # 1.  1)  1:  1-
    r"|[\u0660-\u0669]+\s*[\.\):\-]"                                 # ١.  ٢)  (Arabic-Indic digits)
    r"|(?:ال)?سؤال\s*(?:رقم)?\s*[\d\u0660-\u0669]*\s*[:\-]?"        # سؤال / السؤال / سؤال رقم ١
    r"|س\s*[\d\u0660-\u0669]+\s*[:\-]"                               # س1:  (shorthand)
    r"))",
    re.IGNORECASE,
)

# Blank-line fallback, used only when numbered-marker splitting fails.
_BLANK_LINE_SPLIT = re.compile(r"\n\s*\n+")

# Sentence-level last resort: split after a Latin or Arabic question mark.
_QUESTION_MARK_SPLIT = re.compile(r"(?<=[?\u061F])\s+")


# -----------------------------------------------------------------------------
# LLM-based semantic fallback splitter.
#
# The three regex passes above (numbered markers -> blank lines -> '?' marks)
# cover the vast majority of real exam files, and they're cheap: no GPU call,
# instant, deterministic. But some exam files use numbering the regexes don't
# anticipate (e.g. "Part One", lettered "A/B/C" sections, inconsistent mixed
# numbering, or numbering embedded inside a table that PDF extraction mangles).
# For those, instead of silently mis-splitting (one giant "question" or 30
# tiny meaningless fragments), we ask the LLM itself to read the text and
# return the real question boundaries as JSON. This only fires as a LAST
# resort — after all three regex passes have been tried and rejected — so the
# common case never pays the extra GPU latency.
# -----------------------------------------------------------------------------
QUESTION_SPLIT_PROMPT = PromptTemplate(
    input_variables=["text"],
    template=(
        "You are splitting a university exam/document into its individual "
        "questions. Rule-based splitting could not confidently find question "
        "boundaries in this text (unusual numbering, missing markers, or "
        "mixed formatting).\n"
        "Read the text below and return a STRICT JSON array of strings, one "
        "string per question. Copy each question EXACTLY as it appears in "
        "the text — do not paraphrase, do not translate, do not invent new "
        "questions, do not merge separate questions together. If a question "
        "has lettered sub-parts (a, b, c...), keep them together as ONE "
        "array element.\n"
        "Return JSON ONLY — no markdown fences, no explanation, nothing "
        "before or after the array.\n\n"
        "Text:\n---------\n{text}\n---------\n\nJSON array:"
    ),
)


def _llm_split_questions(full_text: str, max_chars: int = 6000) -> list[str]:
    """
    Ask the LLM to find question boundaries directly, as a semantic fallback
    for exam layouts none of the regex passes recognize.

    Truncates to `max_chars`: this is a single generation call (not
    Map-Reduce like the summarizer), so an extremely long exam is capped to
    keep latency and context-window usage bounded. In practice exam files
    that need this fallback are usually short/irregular, not long lecture
    transcripts, so the cap rarely matters.

    Never raises: any failure here (bad JSON, model/generation error) just
    means "no usable split found", and the caller keeps whatever the regex
    passes produced instead of hard-failing the whole /solve-exam request.
    """
    text_for_llm = full_text[:max_chars]
    logger.info(
        f"split_questions: falling back to LLM-based semantic splitting "
        f"({len(text_for_llm)} of {len(full_text)} chars sent)."
    )
    try:
        raw = run_llm(
            QUESTION_SPLIT_PROMPT.format(text=text_for_llm),
            max_new_tokens=800,
            min_new_tokens=8,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
        )
    except Exception:
        logger.exception("split_questions: LLM-based fallback generation failed.")
        return []

    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        logger.warning("split_questions: LLM fallback did not return a JSON array.")
        return []
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        logger.warning("split_questions: LLM fallback returned malformed JSON.")
        return []

    questions = [str(q).strip() for q in parsed if str(q).strip() and len(str(q).strip()) > 10]
    if len(questions) > 1:
        logger.info(f"split_questions: LLM fallback found {len(questions)} question(s).")
    else:
        logger.warning("split_questions: LLM fallback produced <=1 usable question.")
    return questions[:30]


def _split_is_unreliable(questions: list[str], original_len: int) -> bool:
    """
    A split attempt "succeeds" only if it produces more than one fragment
    AND those fragments are reasonably balanced. Without this check, a
    regex pass could technically find 2 "questions" where one fragment is a
    single stray marker and the other is 95% of the whole exam (e.g. only
    the LAST question in the file happened to use a recognizable marker) —
    that's still effectively a failed split and should fall through to the
    next strategy instead of being accepted as-is.
    """
    if len(questions) <= 1:
        return True
    longest = max(len(q) for q in questions)
    return original_len > 0 and longest > 0.8 * original_len


def split_questions(full_text: str) -> list[str]:
    """
    Split raw exam text into individual questions. This is now a "smart"
    multi-stage splitter: cheap, deterministic regex passes are tried first
    (fast, no GPU), and the LLM is only asked to help when every regex pass
    genuinely fails — so the common case (a normally-numbered exam) never
    pays for an extra generation call.

    Stages, in order:
      1. Numbered/lettered markers (Latin + Arabic styles) — QUESTION_SPLIT_PATTERN.
      2. Blank-line-separated paragraphs.
      3. Sentences ending in '?' / '؟'.
      4. LLM-based semantic split (_llm_split_questions) — last resort for
         exam layouts none of the above recognize (e.g. "Part One" / lettered
         A/B/C sections / numbering mangled by PDF extraction).

    Each stage is accepted only if _split_is_unreliable() says it produced a
    genuinely balanced multi-question split (not just "found 2 pieces where
    one piece is basically the whole file"). If every stage fails, we keep
    whichever raw attempt produced the most fragments rather than returning
    nothing, so the caller can still surface a meaningful result/error
    instead of silently producing an empty solved_exam.pdf.
    """
    if not full_text or not full_text.strip():
        return []

    stripped = full_text.strip()
    # Sentinel leading newline so a marker sitting at position 0 (start of
    # the file/exam) still creates a split boundary via the lookahead above.
    normalized = "\n" + stripped

    raw_questions = QUESTION_SPLIT_PATTERN.split(normalized)
    questions = [q.strip() for q in raw_questions if len(q.strip()) > 10]

    if _split_is_unreliable(questions, len(stripped)):
        logger.warning(
            "split_questions: numbered-marker split unreliable/failed — "
            "trying blank-line fallback split."
        )
        fallback = [q.strip() for q in _BLANK_LINE_SPLIT.split(stripped) if len(q.strip()) > 10]

        if not _split_is_unreliable(fallback, len(stripped)):
            questions = fallback
        else:
            logger.warning(
                "split_questions: blank-line fallback also unreliable/failed — "
                "trying sentence-level ('?'/'؟') fallback split."
            )
            sentence_fallback = [
                q.strip() for q in _QUESTION_MARK_SPLIT.split(stripped) if len(q.strip()) > 10
            ]

            if not _split_is_unreliable(sentence_fallback, len(stripped)):
                questions = sentence_fallback
            else:
                logger.warning(
                    "split_questions: all regex-based passes unreliable/failed — "
                    "trying LLM-based semantic split."
                )
                llm_questions = _llm_split_questions(stripped)

                if not _split_is_unreliable(llm_questions, len(stripped)):
                    questions = llm_questions
                else:
                    # Nothing produced a confidently balanced split. Rather
                    # than returning nothing (which previously meant a
                    # completely blocked /solve-exam request), keep whichever
                    # attempt found the most fragments — a best-effort answer
                    # beats no answer, and answer_question() is already
                    # designed to fail gracefully per-question if the split
                    # was imperfect.
                    logger.warning(
                        "split_questions: no stage produced a confidently "
                        "balanced split — falling back to the best-effort "
                        "attempt with the most fragments."
                    )
                    candidates = [questions, fallback, sentence_fallback, llm_questions]
                    questions = max(candidates, key=len)

    questions = questions[:30]

    logger.info(f"split_questions: detected {len(questions)} question(s).")
    for i, q in enumerate(questions, start=1):
        preview = q[:120].replace("\n", " ")
        logger.info(f"  Q{i}: {preview}{'...' if len(q) > 120 else ''}")

    return questions


def _dedupe_repeated_lines(text: str) -> str:
    """
    Defensive cleanup for repetition loops: collapse consecutive duplicate
    lines/bullets. `no_repeat_ngram_size` blocks exact n-gram repeats within
    the model's own vocabulary window, but a short bullet (e.g. a 3-4 word
    line) can still legally repeat verbatim on the next line without
    tripping it — this catches that case as a second line of defense.
    """
    lines = [l.strip() for l in text.split("\n")]
    deduped: list[str] = []
    for line in lines:
        if line and deduped and line == deduped[-1]:
            continue
        deduped.append(line)
    return "\n".join(deduped)


def _truncate_answer(text: str, max_lines: int = 5) -> str:
    """
    Hard safety net for the '2-5 lines' requirement. The prompt asks the
    model to stop on its own, but instruction-following isn't guaranteed —
    this guarantees the constraint regardless of what the model generates.
    """
    lines = [l for l in text.split("\n") if l.strip()]
    if len(lines) > max_lines:
        lines = lines[:max_lines]
    return "\n".join(lines).strip()


In [ ]:
# 8c. High-level generation functions used by the FastAPI endpoints.
# -----------------------------------------------------------------------------
def summarize_map_reduce(session_id: str) -> str:
    """
    Summarize the ENTIRE course, not just the top-5 chunks a similarity
    search happens to retrieve:
      1. MAP    — every chunk stored for this session is summarized in
                  batches (no FAISS search involved — we already have every
                  chunk from ingest, so we reuse it directly).
      2. REDUCE — the batch summaries are merged into one final,
                  de-duplicated Smart Summary.
    Results are cached on the session so generate_important_topics() can
    reuse them instead of re-running the whole Map step (avoids duplicate
    LLM calls).
    """
    session = get_session(session_id)
    documents = session["documents"]
    batches = batch_documents(documents, max_chars=3000)

    logger.info(f"Map-reduce summary: {len(documents)} chunks -> {len(batches)} batch(es) (session={session_id})")

    partial_summaries = []
    for i, batch in enumerate(batches, start=1):
        context = join_context(batch, max_chars=3000)
        partial = run_llm(CHUNK_SUMMARY_PROMPT.format(context=context))
        partial_summaries.append(partial)
        logger.info(f"Map step {i}/{len(batches)} complete (session={session_id})")

    merged_context = "\n\n".join(f"- {s}" for s in partial_summaries)
    logger.info(f"Reducing {len(partial_summaries)} partial summaries into final summary (session={session_id})")
    final_summary = run_llm(SMART_SUMMARY_PROMPT.format(context=merged_context))

    session["last_summary"] = final_summary
    session["last_partial_summaries"] = partial_summaries
    return final_summary


def generate_smart_summary(session_id: str) -> str:
    return summarize_map_reduce(session_id)


def generate_important_topics(session_id: str) -> list[str]:
    session = get_session(session_id)
    if "last_partial_summaries" not in session:
        # Make sure we have full-course coverage before extracting topics.
        summarize_map_reduce(session_id)
    context = "\n".join(session["last_partial_summaries"])
    logger.info(f"Generating important topics (session={session_id})")
    raw = run_llm(IMPORTANT_TOPICS_PROMPT.format(context=context))
    topics = parse_json_topics(raw)
    session["last_topics"] = topics
    return topics


def generate_revision_notes(session_id: str, summary: str, topics: list[str]) -> str:
    logger.info(f"Generating revision notes (session={session_id})")
    notes = run_llm(REVISION_NOTES_PROMPT.format(summary=summary, topics=", ".join(topics)))
    get_session(session_id)["last_revision_notes"] = notes
    return notes


def answer_question(session_id: str, question: str) -> dict[str, Any]:
    """
    Answer a single exam question against this session's retrieved context.

    FIX: this now (a) uses the shorter EXAM_GENERATION_KWARGS profile instead
    of the 300-token summary profile, directly addressing "answers are too
    long"; (b) runs the output through _dedupe_repeated_lines() +
    _truncate_answer() as a hard safety net for repetition loops / the 2-5
    line limit; and (c) NEVER raises — any generation failure for a single
    question is logged and turned into a graceful fallback answer instead of
    aborting the whole exam. That last point is also the fix for
    solved_exam.pdf sometimes being missing: previously, one bad question
    (a transient generation error, a retrieval edge case, etc.) raised out of
    the list comprehension in solve_exam_file(), aborting /solve-exam before
    export_pdf() was ever called — so NO pdf was produced even though most
    of the exam succeeded.
    """
    session = get_session(session_id)
    # `.get_relevant_documents()` is deprecated in current LangChain in favor
    # of `.invoke()`; using invoke() silences the deprecation warning.
    docs = session["retriever"].invoke(question)
    context = join_context(docs)
    if not context.strip():
        logger.warning(f"Empty retrieval context for a question (session={session_id})")
        context = "No relevant context found in the uploaded material."

    try:
        raw_answer = run_llm(
            SOLVE_EXAM_PROMPT.format(context=context, question=question),
            **EXAM_GENERATION_KWARGS,
        )
        answer = _truncate_answer(_dedupe_repeated_lines(raw_answer), max_lines=5)
        if not answer.strip():
            answer = "The answer was not found in the uploaded material."
        logger.info(
            f"answer_question: generated {len(answer)} char(s) / "
            f"{len(answer.splitlines())} line(s) (session={session_id})"
        )
    except Exception:
        logger.exception(
            f"answer_question: generation failed for question (session={session_id}): {question[:80]!r}"
        )
        answer = "The answer was not found in the uploaded material."

    return {"question": question.strip(), "answer": answer, "confidence": None}


def solve_exam_file(session_id: str, exam_filename: str) -> list[dict[str, Any]]:
    """
    FIX: the original implementation tried to rebuild an exam file's full
    text via `zip(session["raw_texts"], session["filenames"] * 1)` — but
    raw_texts is a list of CHUNKS while filenames is a list of FILES, so the
    lengths never matched and pairs were silently wrong. It then fell back to
    reaching into FAISS's private `vectorstore.docstore._dict`, which is
    fragile across LangChain versions. Both paths could return an empty or
    wrong `full_text`, which is why solve_exam sometimes produced an empty
    solved_exam.pdf or failed outright.

    Fix: `file_texts` (filename -> full raw text) is now stored directly on
    the session at ingest time (see build_session_index), so lookup here is
    a simple, reliable dict access.

    Also: each question is now solved via answer_question(), which never
    raises (see above) — so a problem with one question can no longer take
    down the whole request. GPU memory is freed every few questions, since a
    long exam (20-30+ questions) generating back-to-back was the other main
    trigger for a mid-request CUDA OOM that aborted /solve-exam before
    export_pdf() ran.
    """
    session = get_session(session_id)
    file_texts = session.get("file_texts", {})

    full_text = file_texts.get(exam_filename)
    if full_text is None:
        available = ", ".join(file_texts.keys()) or "none"
        logger.error(f"solve_exam_file: '{exam_filename}' not found in session={session_id} (available: {available})")
        raise KeyError(f"Exam file '{exam_filename}' was not found in this session. Uploaded files: {available}")

    questions = split_questions(full_text)
    if not questions:
        logger.warning(f"solve_exam_file: no questions detected in '{exam_filename}' (session={session_id})")
        raise ValueError(
            f"No questions could be detected in '{exam_filename}'. "
            "Supported numbering styles: '1.', '1)', 'Q1', 'Question 1', 'السؤال 1'."
        )

    logger.info(f"solve_exam_file: solving {len(questions)} question(s) from '{exam_filename}' (session={session_id})")

    solved: list[dict[str, Any]] = []
    for i, q in enumerate(questions, start=1):
        solved.append(answer_question(session_id, q))
        if DEVICE == "cuda" and i % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    return solved


## 9. PDF export

In [ ]:
# 9a. Export any generated text as a downloadable, formatted PDF via ReportLab.
# -----------------------------------------------------------------------------
def _pdf_styles():
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="Heading2Custom", parent=styles["Heading2"], spaceBefore=12, spaceAfter=6))
    styles.add(ParagraphStyle(name="BulletCustom", parent=styles["BodyText"], leftIndent=14, spaceAfter=4))
    return styles


def _is_heading(line: str) -> bool:
    stripped = line.strip().strip("*")
    if stripped.startswith("#"):
        return True
    if stripped.endswith(":") and len(stripped.split()) <= 8:
        return True
    if re.match(r"^\d+\.\s+[A-Z]", stripped) and len(stripped) < 60 and not stripped.endswith("."):
        return True
    return False


def _is_bullet(line: str) -> bool:
    return bool(re.match(r"^\s*[-\u2022*]\s+", line))


def export_pdf(session_id: str, file_type: str, content: str) -> Path:
    """
    Render generated text (summary / revision notes / solved exam) as a
    formatted PDF: title, headings, bullet lists, wrapped paragraphs, and
    page breaks for long content. Every text fragment is XML-escaped before
    being handed to ReportLab's Paragraph, which otherwise interprets raw
    '&', '<', '>' characters as markup and raises mid-build — this was a
    direct cause of missing solved_exam.pdf files whenever exam/lecture text
    contained those characters (e.g. "A < B", "R&D", HTML-like snippets).
    """
    out_dir = OUTPUT_DIR / session_id
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = out_dir / f"{file_type}.pdf"

    logger.info(f"PDF export started: file_type='{file_type}' session={session_id} ({len(content or '')} chars) -> {pdf_path}")

    styles = _pdf_styles()
    story = [Paragraph(xml_escape(file_type.replace("_", " ").title()), styles["Title"]), Spacer(1, 16)]

    if not content or not content.strip():
        story.append(Paragraph("(No content was generated.)", styles["BodyText"]))
    else:
        bullet_buffer: list[str] = []

        def flush_bullets():
            if bullet_buffer:
                items = [ListItem(Paragraph(xml_escape(b), styles["BulletCustom"])) for b in bullet_buffer]
                story.append(ListFlowable(items, bulletType="bullet", start="circle"))
                bullet_buffer.clear()

        for raw_line in content.split("\n"):
            line = raw_line.strip()
            if not line:
                continue
            if line in ("\f", "<pagebreak>"):
                flush_bullets()
                story.append(PageBreak())
                continue
            if _is_bullet(line):
                bullet_buffer.append(re.sub(r"^\s*[-\u2022*]\s+", "", line))
                continue
            flush_bullets()
            if _is_heading(line):
                story.append(Paragraph(xml_escape(line.lstrip("#").strip()), styles["Heading2Custom"]))
            else:
                story.append(Paragraph(xml_escape(line), styles["BodyText"]))
                story.append(Spacer(1, 6))
        flush_bullets()

    try:
        doc = SimpleDocTemplate(
            str(pdf_path), pagesize=A4,
            leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm, bottomMargin=2 * cm,
        )
        doc.build(story)
    except Exception as exc:
        logger.exception(f"PDF export failed: file_type='{file_type}' session={session_id}")
        raise RuntimeError(f"Failed to export PDF '{file_type}': {exc}") from exc

    exists = pdf_path.exists()
    logger.info(f"PDF export completed: file_type='{file_type}' session={session_id} path={pdf_path} exists={exists}")

    if not exists:
        logger.error(f"PDF export reported no error but file is missing: {pdf_path}")
        raise RuntimeError(f"PDF export reported success but file was not created: {pdf_path}")

    logger.info(f"Exported PDF '{pdf_path}' ({pdf_path.stat().st_size} bytes)")
    return pdf_path


## 10. FastAPI app exposed to the local backend (via ngrok)

In [ ]:
# 10a. Define the FastAPI app + endpoints matching what backend/app/services/kaggle_client.py expects.
# -----------------------------------------------------------------------------
kaggle_app = FastAPI(title="Da7ee7-El-Dof3a AI Service")

ALLOWED_DOWNLOAD_TYPES = {"summary", "revision_notes", "solved_exam"}


def _error_response(status_code: int, message: str, **extra) -> JSONResponse:
    logger.error(f"API error ({status_code}): {message}")
    return JSONResponse(status_code=status_code, content={"status": "error", "detail": message, **extra})


@kaggle_app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": SELECTED_MODEL_NAME,
        "quantization": QUANT_MODE,
        "device": DEVICE,
        "active_sessions": len(SESSIONS),
    }


@kaggle_app.post("/ingest")
async def ingest(session_id: str = Form(...), files: list[UploadFile] = File(...)):
    if not files:
        return _error_response(400, "No files were provided.")

    session_dir = UPLOADS_DIR / session_id
    session_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Ingest requested: session={session_id}, {len(files)} file(s)")

    saved_paths = []
    try:
        for f in files:
            dest = session_dir / f.filename
            with dest.open("wb") as out:
                shutil.copyfileobj(f.file, out)
            saved_paths.append(dest)
    except Exception as exc:
        logger.exception(f"Failed saving uploaded files for session={session_id}")
        return _error_response(500, f"Failed to save uploaded files: {exc}")

    try:
        build_session_index(session_id, saved_paths)
    except ValueError as exc:
        return _error_response(400, str(exc))
    except Exception as exc:
        logger.exception(f"Failed to build FAISS index for session={session_id}")
        return _error_response(500, f"Failed to index uploaded files: {exc}")

    return {"session_id": session_id, "status": "indexed", "files": [p.name for p in saved_paths]}


@kaggle_app.post("/generate-summary")
async def generate_summary_endpoint(payload: dict):
    session_id = payload.get("session_id")
    if not session_id:
        return _error_response(400, "session_id is required.")

    try:
        get_session(session_id)
    except KeyError as exc:
        return _error_response(404, str(exc))

    try:
        summary = generate_smart_summary(session_id)
        topics = generate_important_topics(session_id)
        revision = generate_revision_notes(session_id, summary, topics)

        export_pdf(session_id, "summary", summary)
        export_pdf(session_id, "revision_notes", revision)
    except Exception as exc:
        logger.exception(f"generate-summary failed for session={session_id}")
        return _error_response(500, f"Failed to generate summary: {exc}")

    return {
        "session_id": session_id,
        "smart_summary": summary,
        "important_topics": topics,
        "revision_notes": revision,
        "pdf_url": f"/download?session_id={session_id}&file_type=summary",
        "revision_pdf_url": f"/download?session_id={session_id}&file_type=revision_notes",
    }


@kaggle_app.post("/solve-exam")
async def solve_exam_endpoint(payload: dict):
    session_id = payload.get("session_id")
    exam_filename = payload.get("exam_filename")
    question_text = payload.get("question_text")

    if not session_id:
        return _error_response(400, "session_id is required.")

    try:
        get_session(session_id)
    except KeyError as exc:
        return _error_response(404, str(exc))

    if not question_text and not exam_filename:
        return _error_response(400, "Provide 'exam_filename' or 'question_text'.")

    try:
        if question_text:
            logger.info(f"solve-exam: answering a single free-text question (session={session_id})")
            solved = [answer_question(session_id, question_text)]
        else:
            solved = solve_exam_file(session_id, exam_filename)
    except KeyError as exc:
        return _error_response(404, str(exc))
    except ValueError as exc:
        # e.g. "no questions detected in this file"
        return _error_response(422, str(exc))
    except Exception as exc:
        logger.exception(f"solve-exam failed for session={session_id}")
        return _error_response(500, f"Failed to solve exam: {exc}")

    logger.info(f"solve-exam: {len(solved)} question(s) answered (session={session_id})")

    try:
        combined_text = "\n\n".join(f"Q{i + 1}. {q['question']}\n{q['answer']}" for i, q in enumerate(solved))
        pdf_path = export_pdf(session_id, "solved_exam", combined_text)
        logger.info(
            f"solve-exam: solved_exam.pdf ready for session={session_id} "
            f"at {pdf_path} (exists={pdf_path.exists()})"
        )
    except Exception as exc:
        logger.exception(f"Failed to export solved_exam.pdf for session={session_id}")
        return _error_response(500, f"Answers were generated but PDF export failed: {exc}")

    return {
        "session_id": session_id,
        "solved_questions": solved,
        "pdf_url": f"/download?session_id={session_id}&file_type=solved_exam",
    }


@kaggle_app.get("/download")
async def download_endpoint(session_id: str, file_type: str):
    if file_type not in ALLOWED_DOWNLOAD_TYPES:
        return _error_response(400, f"Invalid file_type '{file_type}'. Must be one of: {sorted(ALLOWED_DOWNLOAD_TYPES)}")

    pdf_path = OUTPUT_DIR / session_id / f"{file_type}.pdf"
    if not pdf_path.exists():
        logger.warning(f"Download requested but missing: session={session_id} file_type={file_type}")
        return _error_response(
            404,
            f"'{file_type}.pdf' has not been generated yet for this session. "
            f"Call /generate-summary or /solve-exam first.",
        )

    logger.info(f"Serving download: session={session_id} file_type={file_type}")
    return FileResponse(str(pdf_path), media_type="application/pdf", filename=f"{file_type}.pdf")


@kaggle_app.exception_handler(Exception)
async def unhandled_exception_handler(request: Request, exc: Exception):
    # Safety net: guarantees the API NEVER returns a bare, undiagnosable
    # HTTP 500 — every failure reaches the client as structured JSON.
    logger.exception(f"Unhandled exception on {request.method} {request.url.path}")
    return JSONResponse(status_code=500, content={"status": "error", "detail": f"Internal server error: {exc}"})


## 11. ngrok tunnel — expose the Kaggle FastAPI server publicly

In [22]:
# 11a. Authenticate ngrok using a Kaggle secret / environment variable, then open the tunnel.
# -----------------------------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "NGROK_AUTH_TOKEN not found. Add it as a Kaggle secret or environment variable."
    )

ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()  # ensure no stale tunnels
public_tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = public_tunnel.public_url

print("=" * 60)
print(f"Da7ee7-El-Dof3a AI Service is live at: {PUBLIC_URL}")
print("Paste this URL into backend/.env as KAGGLE_AI_BASE_URL")
print("=" * 60)

2026-07-30 13:29:28 | INFO     | pyngrok.ngrok | Opening tunnel named: http-8000-571e8521-52c0-4b1e-8cd6-090ba3f526a8


2026-07-30 13:29:28 | INFO     | pyngrok.process | Overriding default auth token
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="no configuration paths supplied"
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-07-30 13:29:28 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:28+0000 lvl=info msg="client session established" obj=tunnels.session
2026-07-30 13:

Da7ee7-El-Dof3a AI Service is live at: https://shudder-reviver-undercoat.ngrok-free.dev
Paste this URL into backend/.env as KAGGLE_AI_BASE_URL


2026-07-30 13:29:29 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:29:29+0000 lvl=info msg=end pg=/api/tunnels id=7866adb3526b8011 status=201 dur=151.484446ms


In [23]:
# 11b. Run the FastAPI server in a background thread so the notebook stays interactive.
# -----------------------------------------------------------------------------
def run_server():
    uvicorn.run(kaggle_app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
logger.info(f"FastAPI server running locally on :8000 and publicly at {PUBLIC_URL}")
logger.info("Keep this notebook session running while the backend is in use.")


INFO:     Started server process [298]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-07-30 13:29:38 | INFO     | da7ee7_ai_service | FastAPI server running locally on :8000 and publicly at https://shudder-reviver-undercoat.ngrok-free.dev
2026-07-30 13:29:38 | INFO     | da7ee7_ai_service | Keep this notebook session running while the backend is in use.
2026-07-30 13:35:06 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:35:06+0000 lvl=info msg="join connections" obj=join id=a3d22b23cd09 l=127.0.0.1:8000 r=156.221.132.49:42522
2026-07-30 13:35:08 | INFO     | da7ee7_ai_service | Ingest requested: session=3d503332b2a8, 3 file(s)
2026-07-30 13:35:08 | INFO     | da7ee7_ai_service | Loading 'data_structures_slides.pptx' (type=.pptx)
2026-07-30 13:35:08 | INFO     | da7ee7_ai_service | Loaded 'data_structures_slides.pptx': 1321 characters extracted.
2026-07-30 13:35:08 | INF

INFO:     156.221.132.49:0 - "POST /ingest HTTP/1.1" 200 OK


2026-07-30 13:35:11 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:35:11+0000 lvl=info msg="join connections" obj=join id=43b2afff0545 l=127.0.0.1:8000 r=156.221.132.49:45353
2026-07-30 13:35:11 | INFO     | da7ee7_ai_service | Map-reduce summary: 8 chunks -> 3 batch(es) (session=3d503332b2a8)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=16) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-07-30 13:35:34 | INFO     | da7ee7_ai_service | Map step 1/3 complete (session=3d503332b2a8)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take prec

INFO:     156.221.132.49:0 - "POST /generate-summary HTTP/1.1" 200 OK


2026-07-30 13:37:16 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:37:16+0000 lvl=info msg="join connections" obj=join id=b0cd12c24cb9 l=127.0.0.1:8000 r=156.221.132.49:45373
2026-07-30 13:37:16 | WARNING  | da7ee7_ai_service | split_questions: numbered-marker split unreliable/failed — trying blank-line fallback split.
2026-07-30 13:37:16 | WARNING  | da7ee7_ai_service | split_questions: blank-line fallback also unreliable/failed — trying sentence-level ('?'/'؟') fallback split.
2026-07-30 13:37:16 | INFO     | da7ee7_ai_service | split_questions: detected 4 question(s).
2026-07-30 13:37:16 | INFO     | da7ee7_ai_service |   Q1: Computer Networks — Practice Exam Answer all questions. Total: 9 questions. 1. Explain the difference between the OSI mo...
2026-07-30 13:37:16 | INFO     | da7ee7_ai_service |   Q2: Name its two main protocols. 3. Describe how a three-way handshake establishes a TCP connection. 4. Compare TCP and UDP ...
2026-07-30 13:37:16 | INFO     | da7ee7_ai_service

INFO:     156.221.132.49:0 - "POST /solve-exam HTTP/1.1" 200 OK


2026-07-30 13:37:59 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:37:59+0000 lvl=info msg="join connections" obj=join id=45d9381bc445 l=127.0.0.1:8000 r=156.221.132.49:45389
2026-07-30 13:37:59 | INFO     | da7ee7_ai_service | Serving download: session=3d503332b2a8 file_type=summary


INFO:     156.221.132.49:0 - "GET /download?session_id=3d503332b2a8&file_type=summary HTTP/1.1" 200 OK


2026-07-30 13:44:04 | INFO     | pyngrok.process.ngrok | t=2026-07-30T13:44:04+0000 lvl=info msg="join connections" obj=join id=d6e787f51cf5 l=127.0.0.1:8000 r=156.221.132.49:45629
2026-07-30 13:44:04 | ERROR    | da7ee7_ai_service | Unknown session_id '1f0c89f497a8' requested.
2026-07-30 13:44:04 | ERROR    | da7ee7_ai_service | API error (404): "Unknown session_id '1f0c89f497a8'. Upload files first."


INFO:     156.221.132.49:0 - "POST /generate-summary HTTP/1.1" 404 Not Found


## Notes

- **Gated models:** Llama and Gemma checkpoints require accepting their
  license on Hugging Face and providing an `HF_TOKEN` secret. If a model
  fails to load, the comparison step skips it and continues with the rest.
- **Session persistence:** `SESSIONS` and the FAISS indexes live in this
  notebook's memory/disk for the duration of the Kaggle session. Restarting
  the notebook clears them — students would need to re-upload.
- **Re-running after a Kaggle restart:** ngrok assigns a new public URL each
  time. Update `KAGGLE_AI_BASE_URL` in the backend's `.env` file accordingly.